# Predicción de Demanda y Sistema Inteligente de Alertas de Inventario

## Planteamiento del Problema
En la gestión diaria de un depósito o almacén, los conteos de inventario "a ciegas" o preventivos consumen tiempo y recursos valiosos. Además, el desfasaje constante entre el stock del sistema y la realidad física (debido a ventas en tiempo real, mermas o errores) dificulta la toma de decisiones al momento de realizar pedidos a proveedores.

## Objetivo del Proyecto
El objetivo principal es analizar un dataset histórico de ventas para entrenar un modelo de Machine Learning (Scikit-Learn) capaz de predecir la demanda futura. El fin último es utilizar estas predicciones para simular un sistema de alertas que indique qué productos específicos requieren un conteo físico, optimizando así el tiempo del personal.

## Fases del Proyecto a evaluar:
1. **Análisis Exploratorio (EDA):** Identificar patrones de estacionalidad, tendencias de ventas y comparar el volumen de salida entre distintas tiendas y artículos.
2. **Ingeniería de Características:** Descomponer variables temporales (fechas) para alimentar al algoritmo predictivo.
3. **Modelado y Predicción:** Entrenar un modelo de regresión para pronosticar las ventas a corto plazo.
4. **Lógica de Negocio (Alertas):** Establecer la lógica para detectar discrepancias entre la demanda proyectada y el stock simulado.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## 1. Preprocesamiento

In [ ]:
df = pd.read_csv('train.csv')

print('--- INFORMACIÓN DEL DATASET ---')
df.info()

print('\n--- PRIMERAS 5 FILAS ---')
display(df.head())

--- INFORMACIÓN DEL DATASET ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    913000 non-null  object
 1   store   913000 non-null  int64 
 2   item    913000 non-null  int64 
 3   sales   913000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 27.9+ MB

--- PRIMERAS 5 FILAS ---


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


Transformo la columna "date" a tipo *datetime*. Si Python lee la columna como tipo *object* la va a leer como texto plano, lo cual sería un inconveniente si se decidiera acomodar las fechas de cierta manera o calcular de manera rápida si un producto no se estuvo vendiendo.

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek

display(df.head())

,date,store,item,sales,year,month,day,day_of_week
0,2013-01-01,1,1,13,2013,1,1,1
1,2013-01-02,1,1,11,2013,1,2,2
2,2013-01-03,1,1,14,2013,1,3,3
3,2013-01-04,1,1,13,2013,1,4,4
4,2013-01-05,1,1,10,2013,1,5,5


## 2. Análisis Exploratio de Datos (EDA)

### 2.1. Volumen de Ventas Totales por Tienda
Antes de predecir el futuro, necesitamos entender el volumen histórico. El objetivo de este gráfico es identificar si todas las sucursales manejan un volumen de demanda similar o si existen "tiendas principales" que requerirán mayor atención de stock.

In [ ]:
ventas_por_tienda = df.groupby('store')['sales'].sum().reset_index()

ventas_por_tienda['store'] = ventas_por_tienda['store'].astype(str)

fig_tiendas = px.bar(
    ventas_por_tienda,
    template = 'plotly_dark',
    x = 'store',
    y = 'sales',
    title = '📦 Volumen Histórico de Ventas Totales por Tienda 🏬',
    labels = {'store': 'Número de Tienda 🏬', 'sales': 'Ventas Totales 🛒​'},
    color = 'sales',
    color_continuous_scale = 'Teal'
)

fig_tiendas.update_traces(
    hovertemplate = 'Número de Tienda 🏬=%{x}<br>Ventas Totales 🛒=%{y:,.0f}<extra></extra>'
)

fig_tiendas.update_layout(
    separators = ',.',
    title_x = 0.5,
    title_font = dict(size = 25),
    xaxis_title_font = dict(size = 20),
    yaxis_title_font = dict(size = 20),
    yaxis = dict(ticksuffix = ' u.')
)

fig_tiendas.show()